<a href="https://colab.research.google.com/github/airenas/colab/blob/master/egs/liepa3/ASR/transcribe_zipformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

This colab notebooks shows how to use Python APIs of [sherpa-onnx](https://github.com/k2-fsa/sherpa-onnx) for speech recongition.

# Install sherpa-onnx

In [ ]:
! pip install sherpa-onnx huggingface_hub soundfile

# Download model

## Login to HuggingFace

In [ ]:
from huggingface_hub import login

login()
print("\nDONE: login")

## Load model

Model from huggingface_hub here is trained on Lithuanian 450h corpus

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path
import sherpa_onnx

#################################################################
# Download model
#################################################################
def download_files(tag: str) -> (str, str):
    repo_dir = snapshot_download(tag)
    repo_path = Path(repo_dir)
    # return (str(repo_path / "model/model.onnx"), str(repo_path / "model/tokens.txt"))
    return (str(repo_path / "model/model.int8.onnx"), str(repo_path / "model/tokens.txt"))
    

model_tag_hf= "Airenas/liepa3.zipformer-ctc.v01"
model, tokens = download_files(model_tag_hf)
# model, tokens="models/model.onnx", "models/tokens.txt"
#################################################################
# Load model into memoryt
#################################################################

recognizer = sherpa_onnx.OfflineRecognizer.from_zipformer_ctc(
            model=model,
            tokens=tokens,
            debug=True,
        )
print("MODEL LOADED: READY to transcribe")

## Select audio

Run and then select wav mono 16kHz

In [ ]:
from ipywidgets import FileUpload
from IPython.display import display, Audio
import io
import soundfile as sf

uploader = FileUpload(accept='.wav', multiple=False)
display(uploader)


# Transcribe audio

In [ ]:
#################################################################
# Take file from component
#################################################################
val = uploader.value
if isinstance(val, dict):
    uploaded_files = list(val.values())
elif isinstance(val, (list, tuple)):
    uploaded_files = val
else:
    raise RuntimeError(f"Unexpected uploader.value type: {type(val)}")

if len(uploaded_files) == 0:
    raise RuntimeError(f"Select audio Unexpected uploader.value type: {type(val)}")
uploaded_file = uploaded_files[0]    
#################################################################
# RECOGNIZE
#################################################################
audio_bytes = uploaded_file['content']

samples, sample_rate = sf.read(io.BytesIO(audio_bytes))
print("Sample rate:", sample_rate)
display(Audio(samples, rate=sample_rate))

s = recognizer.create_stream()
s.accept_waveform(sample_rate, samples)
recognizer.decode_stream(s)

print(f"====================================\nResult: {s.result.text}")
print(f"====================================\nFull result: \n {s.result}")

